# ATLAS — NYC Taxi Trip Duration EDA

This notebook documents the saved full `train.csv` scan used to build the dashboard. It runs from the processed data and profile even when the raw CSV is absent. The final app uses only `data/processed/demand_aggregated.csv`; `test.csv` and `sample_submission.csv` are excluded from all metrics.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_path = ROOT / "data" / "train.csv"
processed_path = ROOT / "data" / "processed" / "demand_aggregated.csv"
profile = json.loads((ROOT / "data" / "processed" / "data_profile.json").read_text())
aggregate = pd.read_csv(processed_path, parse_dates=["date"])

## 1. Source shape, fields and missingness

The complete training file has **1,458,644 rows × 11 columns**. Every source column has **0 missing values**. The first and last pickup timestamps are **2016-01-01 00:00:17** and **2016-06-30 23:59:39**. Each row is a recorded trip, not a complete measure of market demand.

In [ ]:
if raw_path.exists():
    display(pd.read_csv(raw_path, nrows=5))
else:
    print("Raw train.csv was removed for deployment; field types and missing counts come from the saved full-source profile.")
display(pd.DataFrame({"source_dtype": profile["columns"], "missing": profile["missing_values"]}))

## 2. Spatial validation

NYC operating bounds are latitude **40.45–40.95**, longitude **-74.3–-73.65**. There are **0** invalid world coordinates and **247** pickups outside this deliberately broad NYC box. We retain **1,458,397** pickups. Among retained pickups, the 1st–99th percentile latitude range is **40.645–40.807**, and longitude range is **-74.014–-73.782**. The broad box may include nearby non-NYC places and excludes distant airport/outlier trips; the dashboard counts only pickups inside it.

In [ ]:
zones = aggregate.groupby(["grid_lat", "grid_lon"], as_index=False)["trip_count"].sum()
fig = px.scatter(zones, x="grid_lon", y="grid_lat", size="trip_count", color="trip_count",
                 color_continuous_scale="Blues", size_max=18, opacity=.55,
                 title="Pickup concentration by 0.01° grid (full retained train set)")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

## 3. Time patterns

The highest single hour is **18:00–19:00**, with **90,585 recorded pickups** across the study period. The retained records include **1,042,227 weekday** and **416,170 weekend** pickups. These are counts across unequal numbers of days, so they should not be interpreted as per-day rates.

In [ ]:
hourly = aggregate.groupby("hour")["trip_count"].sum().reindex(range(24), fill_value=0)
fig = px.line(x=hourly.index, y=hourly.values, markers=True,
              labels={"x":"Hour of day", "y":"Recorded pickups"}, title="Hourly pickup profile")
fig.show()
day_type = aggregate.groupby("day_type")["trip_count"].sum().reset_index()
px.bar(day_type, x="day_type", y="trip_count", color="day_type",
       labels={"trip_count":"Recorded pickups", "day_type":"Day type"},
       title="Weekday vs weekend totals").show()

## 4. Duration and date anomalies

**0** pickup datetimes failed parsing. **0** trips have nonpositive duration; **4** exceed 24 hours. Duration is not used for the pickup demand map, so duration extremes are reported but not removed. The only removed source rows are the **247** with an invalid timestamp or pickup outside the NYC bounds.

## 5. Deployment aggregate

The reproducible pipeline (`python -m src.data_processing`) groups retained trips by **date, hour, day type and 0.01° pickup grid**. It produces **239,648 rows** in a **9.31 MB** CSV. Aggregation preserves the count of all retained pickups and supports daily filtering without loading the 200.59 MB raw CSV in Streamlit. Grid centers are approximate; they are not administrative neighborhood labels.

In [ ]:
assert aggregate["trip_count"].sum() == profile["retained_pickups"]
assert len(aggregate) == profile["aggregate_rows"]
print("Aggregate checks passed:", aggregate["trip_count"].sum(), "pickups")